# Module 3: Wednesday Session
## CS 5045: Computation for the Data Sciences
### Conditionals & Control Flow: Live Demo

This notebook walks through Filter and Transform using the student performance dataset introduced in Module 3.
Work through each section in order. 

---
## Setup

We load the dataset using the `csv` module (no pandas yet: we want to see the raw loop-and-if mechanics).
Each row becomes a dictionary: keys are column names, values are the cell values.

In [ ]:
import csv

students = []
with open('data.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        row['attendance_pct']       = float(row['attendance_pct'])
        row['assignment_avg']       = float(row['assignment_avg'])
        row['midterm_score']        = float(row['midterm_score'])
        row['final_score']          = float(row['final_score'])
        row['study_hours_per_week'] = int(row['study_hours_per_week'])
        students.append(row)

print(f'Loaded {len(students)} student records.')
print('Columns:', list(students[0].keys()))
print()
print('First row:', students[0])

---
## REPRESENT

We have just performed the **Represent** operation: 16 rows of student data now live in memory as a list of dictionaries.

Before we Filter or Transform, we need something to work with. Represent always comes first.

Look at the first row printed above. What type is `attendance_pct`? What type is `major`? Why does the distinction matter before we write a conditional?

---
## TRANSFORM: Assign Letter Grades

**Operation:** Transform: produce a new value (`letter_grade`) for every row.

Strategy:
1. Define a function that encapsulates the grading rule.
2. Use a `for` loop to apply the function to every student.

In [ ]:
def assign_grade(score):
    """Return a letter grade for a numeric score."""
    if score >= 90:
        return 'A'
    elif score >= 80:
        return 'B'
    elif score >= 70:
        return 'C'
    elif score >= 60:
        return 'D'
    else:
        return 'F'

# Apply the Transform to every student
for student in students:
    student['letter_grade'] = assign_grade(student['final_score'])

print(f'{'Name':<22} {'Final':>7} {'Grade':>6}')
print('-' * 38)
for s in students:
    print(f"{s['name']:<22} {s['final_score']:>7.1f} {s['letter_grade']:>6}")

Count the rows. We started with 16 and we still have 16. Every student got a new column value. That is Transform.

What would you change to use a 10-point scale (A = 93+, B = 83+, ...)? Where is the one place you would edit?

---
## TRANSFORM: Flag Low Attendance

**Operation:** Transform: a binary condition produces a new categorical value per row.

This is the simplest possible conditional: one threshold, two outcomes.

In [ ]:
ATTENDANCE_THRESHOLD = 75.0

for student in students:
    if student['attendance_pct'] < ATTENDANCE_THRESHOLD:
        student['attendance_flag'] = 'low'
    else:
        student['attendance_flag'] = 'ok'

print(f"{'Name':<22} {'Attend%':>8} {'Flag':>8}")
print('-' * 42)
for s in students:
    marker = '  <-- flagged' if s['attendance_flag'] == 'low' else ''
    print(f"{s['name']:<22} {s['attendance_pct']:>8.1f} {s['attendance_flag']:>8}{marker}")

How many students are flagged? If you changed the threshold to 80%, how many would be flagged? Where would you make that change in the code?

---
## FILTER: Students Who Passed the Midterm

**Operation:** Filter: keep only rows satisfying the condition. Row count changes.

Pattern: empty list → loop → conditional append.

In [ ]:
passed = []                              # start with an empty result
for student in students:                 # visit every row
    if student['passed_midterm'] == 'yes':   # test the condition
        passed.append(student)           # keep the row if True

print(f'Before filter: {len(students)} students')
print(f'After  filter: {len(passed)} students')
print()
print(f"{'Name':<22} {'Midterm':>8}")
print('-' * 33)
for s in passed:
    print(f"{s['name']:<22} {s['midterm_score']:>8.1f}")

The row count changed: we went from 16 to a smaller number. Rows were removed, not modified. That is the defining signature of Filter.

What would you change to filter by `major == 'Biology'` instead? Type your answer in chat.

---
## FILTER: Compound Condition

**Operation:** Filter with `and`: both conditions must be True for a row to be kept.

Adding more conditions does not change the operation type. It is still Filter.

In [ ]:
high_performers = []
for student in students:
    if student['attendance_pct'] >= 85 and student['assignment_avg'] >= 85:
        high_performers.append(student)

print(f'High performers (attendance >= 85% AND assignment avg >= 85): {len(high_performers)}')
print()
print(f"{'Name':<22} {'Attend%':>8} {'Assign Avg':>11}")
print('-' * 44)
for s in high_performers:
    print(f"{s['name']:<22} {s['attendance_pct']:>8.1f} {s['assignment_avg']:>11.1f}")

What if you changed `and` to `or`? Would the result be larger or smaller? Why? Try it.

---
## TRANSFORM: Compute a Composite Score

**Operation:** Transform: a formula (not a conditional) produces a new numeric column per row.

Composite = 30% midterm + 50% final + 20% assignment average.

In [ ]:
for student in students:
    student['composite'] = (
        0.30 * student['midterm_score']
        + 0.50 * student['final_score']
        + 0.20 * student['assignment_avg']
    )

print(f"{'Name':<22} {'Midterm':>8} {'Final':>7} {'Assign':>7} {'Composite':>10}")
print('-' * 58)
for s in sorted(students, key=lambda x: x['composite'], reverse=True):
    print(
        f"{s['name']:<22} {s['midterm_score']:>8.1f} "
        f"{s['final_score']:>7.1f} {s['assignment_avg']:>7.1f} "
        f"{s['composite']:>10.2f}"
    )

Count the rows. Still 16: every student got a new column. Transform.

What would you change the weights to if the course were graded 40% midterm, 40% final, 20% assignments? What must the weights sum to, and why?

---
## Connecting to the Assignment

The assignment this week asks you to apply exactly these patterns to the same dataset:

1. **Transform:** Write an `if`/`elif`/`else` that categorizes each student's attendance as `'high'`, `'medium'`, or `'low'` based on thresholds you choose. Add the result as a new column `'attendance_category'`.

2. **Filter:** Isolate students who meet a combined criterion of your design: any two conditions joined by `and` or `or`. Print both the count before and after, and the names of the students who remain.

3. **Transform:** Compute a composite score with weights you define. Your weights must sum to 1.0. Add the result as `'my_composite'` and sort the output from highest to lowest.

For each part, include a comment in plain English stating what the operation is (Filter or Transform) and why.

**Questions before you go? Post in the Canvas discussion or bring them to office hours.**